# Surface Biomarkers Workflow

Notebook exemplo para usar o pacote `surface_biomarkers` em dados single-cell/snRNA-seq.

Fluxo:

1. carregar matriz de contagem;
2. filtrar genes de superficie ou genes customizados;
3. treinar modelos one-vs-rest;
4. gerar metricas, ROC, matriz de confusao e SHAP;
5. descobrir assinaturas compactas;
6. validar assinaturas com banco de superficie/anticorpos;
7. desenhar painel de anticorpos.

## 1. Instalar em modo editavel

Rode esta celula uma vez. Depois reinicie o kernel se o pacote acabou de ser instalado.

In [ ]:
%pip install -e .

## 2. Imports

In [ ]:
from surface_biomarkers import (
    DiscoveryConfig,
    TrainingConfig,
    design_antibody_panel,
    discover_signatures,
    evaluate_signatures,
    load_counts_csv,
    prioritize_signature_validation,
    save_signature_validation_report,
    train_one_vs_rest,
)
from surface_biomarkers.plots import analyze_clusters, plot_panel_benchmark, plot_signature_accuracy, plot_shap_heatmap
from surface_biomarkers.signatures import (
    combined_shap_expression_matrix,
    mean_abs_shap_matrix,
    validate_signature_specificity,
)

import pandas as pd

## 3. Configuracoes principais

Altere os caminhos e os clusters conforme seu dataset.

In [ ]:
COUNTS_CSV = "counts.csv"
TARGET_COLUMN = "celltype"
OUTPUT_DIR = "cluster_outputs"

clusters = ["0", "1", "2", "3", "4", "5"]

cluster_names = {
    "0": "Mac1",
    "1": "Mac2",
    "2": "Mac3",
    "3": "Mac4",
    "4": "Mac5",
    "5": "Mac6",
}

## 4. Carregar os dados

Use `gene_filter="surface"` para manter genes do banco interno `db_surface.csv`.

Opcoes:

- `gene_filter="all"`: usa todos os genes;
- `gene_filter="surface"`: usa o banco interno de genes de superficie;
- `gene_filter="custom"`: usa uma lista de genes escolhida por voce.

In [ ]:
dataset = load_counts_csv(
    COUNTS_CSV,
    target_column=TARGET_COLUMN,
    gene_filter="surface",
    encode_labels=False,
)

dataset.shape

### Opcional: usar genes customizados

In [ ]:
# genes_interesse = ["PLXND1", "TLR1", "ABCA1", "CXADR", "FN1"]
# dataset = load_counts_csv(
#     COUNTS_CSV,
#     target_column=TARGET_COLUMN,
#     gene_filter="custom",
#     genes=genes_interesse,
# )

## 5. Treinar modelos one-vs-rest

Isso treina um modelo binario para cada cluster: cluster alvo vs todos os outros.

In [ ]:
training_config = TrainingConfig(
    resampling="SMOTE",
    cv=5,
    n_jobs=-1,
    verbose=1,
    grid_verbose=0,
    train_random_forest=True,
    train_lightgbm=True,
)

model_results = train_one_vs_rest(
    dataset,
    target_column=TARGET_COLUMN,
    output_dir=OUTPUT_DIR,
    config=training_config,
)

model_results

## 6. Gerar metricas e figuras por cluster

Gera matriz de confusao, curva ROC, SHAP individual e painel por cluster.

In [ ]:
metrics = analyze_clusters(
    base_dir=OUTPUT_DIR,
    clusters=clusters,
    model_name=["LGBM", "RF"],
    max_display=10,
    cluster_names=cluster_names,
    panel=True,
    show_panel=True,
)

metrics

## 7. Descobrir assinaturas compactas

Seleciona genes importantes por SHAP, incluindo marcadores positivos e negativos/exclusao conforme a direcao SHAP-expressao.

In [ ]:
discovery_config = DiscoveryConfig(
    model_name="LGBM",
    positive_genes=1,
    negative_genes=1,
    min_expression_prop=0.10,
    shap_corr_positive=0.15,
    shap_corr_negative=-0.10,
    top_n_heatmap=10,
)

signatures, gene_ranking, gene_exclusivity = discover_signatures(
    base_dir=OUTPUT_DIR,
    clusters=clusters,
    config=discovery_config,
)

signatures

In [ ]:
gene_ranking.head(20)

In [ ]:
gene_exclusivity.head(20)

## 8. Heatmaps SHAP

In [ ]:
shap_matrix = mean_abs_shap_matrix(OUTPUT_DIR, clusters, discovery_config)
combined_matrix = combined_shap_expression_matrix(OUTPUT_DIR, clusters, discovery_config)

plot_shap_heatmap(
    shap_matrix,
    f"{OUTPUT_DIR}/shap_top_genes_heatmap.svg",
    title="Top genes by mean absolute SHAP",
    cluster_names=cluster_names,
    show=True,
)

plot_shap_heatmap(
    combined_matrix,
    f"{OUTPUT_DIR}/shap_expression_heatmap.svg",
    title="SHAP x expression score",
    cluster_names=cluster_names,
    show=True,
)

## 9. Validar especificidade da assinatura

In [ ]:
specificity = validate_signature_specificity(signatures, combined_matrix)
specificity

## 10. Acuracia independente das assinaturas pequenas

Esta funcao segue a logica original do notebook: soma os genes da assinatura em cada bloco `X_test.csv` e calcula acuracia com `score > threshold`.

In [ ]:
accuracy = evaluate_signatures(
    base_dir=OUTPUT_DIR,
    clusters=clusters,
    signatures=signatures,
    threshold=0.5,
    cluster_names=cluster_names,
)

accuracy

In [ ]:
plot_signature_accuracy(
    accuracy,
    f"{OUTPUT_DIR}/signature_accuracy.svg",
    cluster_names=cluster_names,
    show=True,
)

## 11. Validacao translacional com superficie e anticorpos

Usa os bancos internos:

- `resources/db_surface.csv`
- `resources/proteinatlas.tsv`

Como o modelo foi treinado em snRNA-seq, os resultados sao candidatos para validacao experimental, nao marcadores proteicos validados.

In [ ]:
validation_priority = prioritize_signature_validation(
    ranking=gene_ranking,
    signatures=signatures,
    surface_db="internal",
    protein_atlas="internal",
    signature_only=True,
)

validation_priority.head(20)

In [ ]:
validation_priority = save_signature_validation_report(
    ranking=gene_ranking,
    signatures=signatures,
    surface_db="internal",
    protein_atlas="internal",
    output_path=f"{OUTPUT_DIR}/signature_validation_priority.csv",
    signature_only=True,
)

validation_priority.head()

## 12. Desenhar painel de anticorpos

A funcao abaixo recomenda combinacoes de marcadores, nao apenas genes individuais.

Ela retorna marcadores positivos, marcadores negativos/exclusao, backups e um painel minimo.

In [ ]:
panel_facs = design_antibody_panel(
    markers=validation_priority,
    application="FACS",
    max_positive_markers=2,
    max_negative_markers=1,
    max_backup_markers=2,
)

panel_facs

In [ ]:
panel_cite = design_antibody_panel(
    markers=validation_priority,
    application="CITE-seq",
)

panel_cite

## 13. Benchmark visual do painel

Compara score, tamanho do painel, backups, penalidade de redundancia e constraints por cluster/aplicacao.

In [ ]:
plot_panel_benchmark(
    panel_facs,
    f"{OUTPUT_DIR}/panel_benchmark_FACS.svg",
    cluster_names=cluster_names,
    show=True,
)

## 14. Salvar tabelas principais

In [ ]:
gene_ranking.to_csv(f"{OUTPUT_DIR}/gene_ranking_by_cluster.csv", index=False)
gene_exclusivity.to_csv(f"{OUTPUT_DIR}/gene_exclusivity.csv", index=False)
specificity.to_csv(f"{OUTPUT_DIR}/signature_specificity.csv", index=False)
accuracy.to_csv(f"{OUTPUT_DIR}/signature_accuracy.csv", index=False)
panel_facs.to_csv(f"{OUTPUT_DIR}/antibody_panel_FACS.csv", index=False)
panel_cite.to_csv(f"{OUTPUT_DIR}/antibody_panel_CITEseq.csv", index=False)

plot_panel_benchmark(
    panel_cite,
    f"{OUTPUT_DIR}/panel_benchmark_CITEseq.svg",
    cluster_names=cluster_names,
    show=False,
)